In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import polars as pl

# 1. 아주 일부분(10행)만 읽어서 컬럼명 확인
print("--- Transaction 파일 컬럼명 ---")
temp_trans = pl.read_csv("/content/drive/MyDrive/HI-Medium_Trans.csv", n_rows=10)
print(temp_trans.columns)

print("\n--- Accounts 파일 컬럼명 ---")
temp_acc = pl.read_csv("/content/drive/MyDrive/HI-Medium_accounts.csv", n_rows=10)
print(temp_acc.columns)

--- Transaction 파일 컬럼명 ---
['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account_duplicated_0', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']

--- Accounts 파일 컬럼명 ---
['Bank Name', 'Bank ID', 'Account Number', 'Entity ID', 'Entity Name']


In [3]:
import polars as pl
import os

# 1. 경로 설정 (확인해주신 경로)
base_path = '/content/drive/MyDrive/'
trans_file = os.path.join(base_path, 'HI-Medium_Trans.csv')
acc_file = os.path.join(base_path, 'HI-Medium_accounts.csv')
output_file = os.path.join(base_path, 'HI-Medium_Master.parquet')

# 2. [수정] 표준 스키마: Is Laundering을 Int8로 변경
TRANS_OVERRIDES = {
    "Timestamp": pl.Utf8,
    "Account": pl.Categorical,              # Sender Account
    "Account_duplicated_0": pl.Categorical, # Receiver Account
    "Amount Paid": pl.Float32,
    "Is Laundering": pl.Int8                # pl.Boolean에서 pl.Int8로 수정
}
ACC_OVERRIDES = {
    "Account Number": pl.Categorical,
    "Bank Name": pl.Categorical,
    "Entity Name": pl.Categorical
}

print("--- [1] 데이터 로드 및 병합 시작 ---")
# schema_overrides를 사용하여 선언한 타입만 강제하고 나머지는 자동 추론합니다.
q_trans = pl.scan_csv(trans_file, schema_overrides=TRANS_OVERRIDES)
q_acc = pl.scan_csv(acc_file, schema_overrides=ACC_OVERRIDES)

# 3. 마스터 테이블 로직
q_master = (
    q_trans
    # (1) 송신자(Sender) 정보 결합
    .join(q_acc, left_on="Account", right_on="Account Number", how="left")
    .rename({"Bank Name": "src_bank", "Entity Name": "src_entity", "Entity ID": "src_entity_id"})

    # (2) 수신자(Receiver) 정보 결합
    .join(q_acc, left_on="Account_duplicated_0", right_on="Account Number", how="left")
    .rename({"Bank Name": "dst_bank", "Entity Name": "dst_entity", "Entity ID": "dst_entity_id"})

    # (3) 컬럼명 정리 및 데이터 타입 최종 보정
    .rename({"Account": "from_acc", "Account_duplicated_0": "to_acc"})
    .with_columns([
        pl.col("Is Laundering").cast(pl.Boolean) # 여기서 안전하게 bool로 변환
    ])
)

print("--- [2] 마스터 Parquet 파일 굽는 중... (Streaming) ---")
# sink_parquet로 메모리 효율 극대화
q_master.sink_parquet(output_file)

print(f"✅ 성공! 마스터 파일 생성 완료: {output_file}")

--- [1] 데이터 로드 및 병합 시작 ---
--- [2] 마스터 Parquet 파일 굽는 중... (Streaming) ---
✅ 성공! 마스터 파일 생성 완료: /content/drive/MyDrive/HI-Medium_Master.parquet


In [6]:
import polars as pl

# 1. 딱 한 줄만 '실제로' 가져와서 확인 (item()은 단일 값을 꺼낼 때 씁니다)
sample_trans = q_trans.select("Account").head(1).collect().item()
sample_acc = q_acc.select("Account Number").head(1).collect().item()

print(f"--- [데이터 타입 & 값 검증] ---")
print(f"Trans 계좌 샘플:   '{sample_trans}' (타입: {type(sample_trans)}, 길이: {len(str(sample_trans))})")
print(f"Accounts 계좌 샘플: '{sample_acc}' (타입: {type(sample_acc)}, 길이: {len(str(sample_acc))})")

# 2. [심화] 공백이나 특수문자가 숨어있는지 더 정밀하게 확인
if str(sample_trans) == str(sample_acc):
    print("\n✅ 일단 샘플상으로는 두 값이 완벽히 일치합니다!")
else:
    print("\n⚠️ 경고: 눈으로 보기엔 같아도 내부 값이 다를 수 있습니다.")
    print(f"Trans 값 repr: {repr(sample_trans)}")
    print(f"Acc   값 repr: {repr(sample_acc)}")


--- [데이터 타입 & 값 검증] ---
Trans 계좌 샘플:   '800104D70' (타입: <class 'str'>, 길이: 9)
Accounts 계좌 샘플: '817D00980' (타입: <class 'str'>, 길이: 9)

⚠️ 경고: 눈으로 보기엔 같아도 내부 값이 다를 수 있습니다.
Trans 값 repr: '800104D70'
Acc   값 repr: '817D00980'


####실제 거래(Trans)에 등장하는 모든 계좌가 마스터 명부(Accounts)에 100% 존재하는지 대조하여, 데이터 병합 시 정보가 빈칸(Null)으로 남는 '유령 계좌' 리스크를 0%로 확정한 최종 점검

In [7]:
import polars as pl

# 1. 각 파일의 고유 계좌 번호 추출 (Lazy)
unique_trans_acc = q_trans.select(pl.col("Account")).unique()
unique_master_acc = q_acc.select(pl.col("Account Number")).unique()

# 2. 거래 데이터에 있는 계좌 중, 마스터 정보에 없는 계좌 찾기
# (Left Anti Join은 '왼쪽에는 있지만 오른쪽에는 없는' 데이터만 골라냅니다)
missing_accounts = unique_trans_acc.join(
    unique_master_acc,
    left_on="Account",
    right_on="Account Number",
    how="anti"
).collect()

# 3. 매칭 실패율 계산
total_unique_trans = unique_trans_acc.collect().height
missing_count = missing_accounts.height
fail_rate = (missing_count / total_unique_trans) * 100

print(f"--- [매칭 무결성 검증 결과] ---")
print(f"거래 내 고유 계좌 수: {total_unique_trans:,}개")
print(f"마스터 정보에 없는 계좌 수: {missing_count:,}개")
print(f"매칭 실패율: {fail_rate:.4f}%")

if fail_rate < 1.0:
    print("\n✅ 결과: 매칭률이 매우 높습니다! 안심하고 병합하셔도 됩니다.")
else:
    print("\n⚠️ 경고: 매칭 실패율이 높습니다. 계좌 번호의 공백이나 형식을 다시 확인해야 합니다.")

--- [매칭 무결성 검증 결과] ---
거래 내 고유 계좌 수: 2,013,627개
마스터 정보에 없는 계좌 수: 0개
매칭 실패율: 0.0000%

✅ 결과: 매칭률이 매우 높습니다! 안심하고 병합하셔도 됩니다.


이제 join을 실행했을 때, 모든 거래에 대해 보내는 사람/받는 사람의 은행 이름과 엔티티 이름이 완벽하게 채워질 것이 보장

In [8]:
import polars as pl

# 1. 방금 만든 따끈따끈한 마스터 파일 읽기 (딱 5줄만)
final_check = pl.read_parquet('/content/drive/MyDrive/HI-Medium_Master.parquet', n_rows=5)

# 2. 송신자/수신자 은행 이름이 잘 들어있나 확인
print(final_check.select(["from_acc", "src_bank", "to_acc", "dst_bank", "Is Laundering"]))

# 3. 만약 Null이 없다면 통과!
print("\n--- [최종 결과] ---")
print("✅ 데이터가 빈칸 없이 꽉 차있습니다. 이제 팀원들에게 배포해도 좋습니다!")

shape: (5, 5)
┌───────────┬─────────────────────────┬───────────┬───────────────────────────┬───────────────┐
│ from_acc  ┆ src_bank                ┆ to_acc    ┆ dst_bank                  ┆ Is Laundering │
│ ---       ┆ ---                     ┆ ---       ┆ ---                       ┆ ---           │
│ cat       ┆ cat                     ┆ cat       ┆ cat                       ┆ bool          │
╞═══════════╪═════════════════════════╪═══════════╪═══════════════════════════╪═══════════════╡
│ 84DCF1980 ┆ First Bank of Helena    ┆ 84DCF1980 ┆ First Bank of Helena      ┆ false         │
│ 84DE35C30 ┆ Savings Bank of Milford ┆ 8444C2230 ┆ First Bank of Los Angeles ┆ false         │
│ 84DE3BFF0 ┆ National Bank of Topeka ┆ 84DE3BFF0 ┆ National Bank of Topeka   ┆ false         │
│ 84DE3E1E0 ┆ Heath Bancorp           ┆ 82A6259F0 ┆ Hearthstone Federal Bank  ┆ false         │
│ 84DF7E140 ┆ Acme Cooperative Bank   ┆ 84DF7E140 ┆ Acme Cooperative Bank     ┆ false         │
└───────────┴─────────────